# 02 — VL 4B FP8, enseignant 8B et MSE

**Objectif :** comprendre l'alignement des représentations et ce que les chiffres permettent de conclure.

Les features sont prises avant la dernière RMSNorm, avec le même template et les mêmes tokens. Le 4B FP8 encode un prompt à la fois ; l'enseignant BF16 utilise son propre cache. Une identité de modèle, une révision et une empreinte de corpus lient chaque cache au calcul qui l'a produit.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


## Comment calculer la MSE

Pour une prédiction \(\hat y\), une cible \(y\), et \(N\) composantes valides :

\[\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2\]

L'alignement utilise une MSE relative :

\[\mathrm{MSE}_{relative}=\frac{\operatorname{moyenne}((\hat y-y)^2)}{\max(\operatorname{moyenne}(y^2),10^{-8})}\]

Le score d'évaluation est ensuite moyenné par prompt. La perte d'embeddings ajoute \(0{,}1(1-\cosinus)\). Une valeur de 0,10 n'est **pas** une erreur de position de 10 %.

Le calcul suivant est un exemple pédagogique, **pas une mesure du modèle**.

In [ ]:
predicted = [2.0, 4.0]
target = [1.0, 3.0]
mse = sum((a - b) ** 2 for a, b in zip(predicted, target)) / len(target)
energy = sum(b ** 2 for b in target) / len(target)
print("MSE :", mse)
print("MSE relative :", mse / max(energy, 1e-8))

## Pourquoi le 4B et le 8B restent différents

Le 4B a une capacité et des représentations différentes. La quantification FP8 ajoute une autre source d'écart. Un MLP réduit l'écart d'embeddings, mais ne garantit pas que le diffuseur interprète identiquement les relations spatiales.

La nouvelle chaîne ajoute attention entre tokens, boîtes optionnelles, images supervisées et trajectoires de l'enseignant. Elle mesure ensuite les images, car une faible MSE d'embeddings ne prouve pas le bon nombre d'objets, leur identité ou leur position.

Le contrôle du kernel FP8 compare une opération FP8 à sa référence déquantifiée. Il ne mesure pas la parité générale du modèle 4B avec le 8B.

In [ ]:
audit_path = ARTIFACTS / "runtime-audit.json"
if not audit_path.exists():
    audit_path = ARTIFACTS / "spatial/remote-metadata/runtime-audit.json"
audit = read_json(audit_path)
if audit:
    table([{"champ": key, "valeur": audit.get(key)} for key in (
        "passed", "fp8_linears", "fp8_prefill_policy", "feature_shapes", "scope"
    )], ["champ", "valeur"])
else:
    print("Rapport FP8 non présent dans cette copie ; aucune conclusion inventée.")

## Lire les identités du run final

Les SHA des index, du corpus et du code sont conservés avec le rapport final. Ils permettent d'identifier les features utilisées sans ajouter leurs gros tenseurs au dépôt Git.

In [ ]:
archive = archived_report()
summary = read_json(archive / "summary.json", {}) if archive else {}
identity = summary.get("training_identity", {})
if identity:
    print(__import__("json").dumps(identity, ensure_ascii=False, indent=2))
else:
    print("Identités finales à collecter après la fin du run.")

## Commandes reproductibles

L'audit mobilise le GPU. L'extraction des features est pilotée par la chaîne du chapitre 03 afin de préserver l'ordre des téléchargements et le budget disque. Les poids 8B temporaires peuvent être libérés **seulement** après validation complète de leur cache ; le runner distant gère cette séquence.

L'ancien alignement linéaire + MLP est documenté dans [01-encoder-distillation](../01-encoder-distillation/README.md). Ses résultats restent distincts de ceux du nouveau projecteur spatial.

In [ ]:
RUN_FP8_AUDIT = False
run_script("01-encoder-distillation/audit_runtime.py",
           enabled=RUN_FP8_AUDIT, gpu=True)

**Suite :** [03 — Entraîner la Base spatiale](03_base_spatiale.ipynb).